# Qwen Instruction SFT

本 notebook 用于完成第一个小项目：基于 Qwen 的基础指令微调。

内容包括：
1. 数据加载与检查
2. Chat template 对齐
3. LoRA 训练
4. 两组对比实验
5. 结果图表生成与保存


## 1. Environment Setup

In [ ]:
from pathlib import Path
import json
import math
import random
from dataclasses import dataclass, asdict

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


In [ ]:
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path('/Users/yezibin/Project/learn-ft-and-rl/project/qwen-instruction-sft')
DATA_DIR = PROJECT_DIR / 'data'
RUNS_DIR = PROJECT_DIR / 'runs'
TRAIN_PATH = DATA_DIR / 'train' / 'train_v0.1.jsonl'
EVAL_PATH = DATA_DIR / 'eval' / 'eval_v0.1.jsonl'

BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'
FALLBACK_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

sns.set_theme(style='whitegrid')
PROJECT_DIR, TRAIN_PATH.exists(), EVAL_PATH.exists()

## 2. Load and Inspect Data

In [ ]:
def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
eval_rows = load_jsonl(EVAL_PATH)
len(train_rows), len(eval_rows)

In [ ]:
pd.DataFrame([r['metadata'] for r in train_rows]).value_counts('task_type')

In [ ]:
train_rows[0]

## 3. Tokenizer and Chat Template

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

preview_text = tokenizer.apply_chat_template(
    train_rows[0]['messages'],
    tokenize=False,
    add_generation_prompt=False,
)
print(preview_text)

## 4. Convert Messages to Training Text

In [ ]:
def to_chat_text(row):
    return {
        'id': row['id'],
        'text': tokenizer.apply_chat_template(
            row['messages'],
            tokenize=False,
            add_generation_prompt=False,
        ),
        'task_type': row['metadata']['task_type'],
    }

train_text_rows = [to_chat_text(r) for r in train_rows]
eval_text_rows = [to_chat_text(r) for r in eval_rows]
train_text_rows[0]['text'][:400]

## 5. Build Hugging Face Datasets

In [ ]:
train_ds = Dataset.from_list(train_text_rows)
eval_ds = Dataset.from_list(eval_text_rows)
train_ds, eval_ds

In [ ]:
MAX_LENGTH = 1024

def tokenize_batch(batch):
    out = tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
        padding='max_length',
    )
    out['labels'] = out['input_ids'].copy()
    return out

tokenized_train = train_ds.map(tokenize_batch, batched=True)
tokenized_eval = eval_ds.map(tokenize_batch, batched=True)
tokenized_train[0].keys()

## 6. Experiment Configurations

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    output_dir: str
    lora_r: int
    lora_alpha: int
    lora_dropout: float
    learning_rate: float
    num_train_epochs: int
    per_device_train_batch_size: int
    gradient_accumulation_steps: int

experiments = [
    ExperimentConfig(
        name='exp-001-baseline-lora',
        output_dir=str(RUNS_DIR / 'exp-001'),
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        learning_rate=2e-4,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
    ),
    ExperimentConfig(
        name='exp-002-higher-rank',
        output_dir=str(RUNS_DIR / 'exp-002'),
        lora_r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        learning_rate=2e-4,
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
    ),
]

pd.DataFrame([asdict(exp) for exp in experiments])

## 7. Model Builder

In [ ]:
def build_model_and_tokenizer(model_name=BASE_MODEL, load_in_4bit=False):
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        device_map='auto',
    )
    lora_targets = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'up_proj', 'down_proj', 'gate_proj']
    return model, lora_targets


## 8. Trainer Builder

In [ ]:
def build_trainer(exp_cfg: ExperimentConfig):
    model, lora_targets = build_model_and_tokenizer()
    peft_cfg = LoraConfig(
        r=exp_cfg.lora_r,
        lora_alpha=exp_cfg.lora_alpha,
        lora_dropout=exp_cfg.lora_dropout,
        target_modules=lora_targets,
        bias='none',
        task_type='CAUSAL_LM',
    )
    model = get_peft_model(model, peft_cfg)
    args = TrainingArguments(
        output_dir=exp_cfg.output_dir,
        learning_rate=exp_cfg.learning_rate,
        num_train_epochs=exp_cfg.num_train_epochs,
        per_device_train_batch_size=exp_cfg.per_device_train_batch_size,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=exp_cfg.gradient_accumulation_steps,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        logging_steps=5,
        report_to='none',
        bf16=True,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        tokenizer=tokenizer,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )
    return trainer


## 9. Run Experiments

In [ ]:
# Warning: this cell will start training.
# Uncomment when the environment is ready.

# histories = []
# for exp in experiments:
#     trainer = build_trainer(exp)
#     train_result = trainer.train()
#     eval_result = trainer.evaluate()
#     history = pd.DataFrame(trainer.state.log_history)
#     history['experiment'] = exp.name
#     history.to_csv(Path(exp.output_dir) / 'log_history.csv', index=False)
#     histories.append(history)
#     with open(Path(exp.output_dir) / 'eval_metrics.json', 'w', encoding='utf-8') as f:
#         json.dump(eval_result, f, ensure_ascii=False, indent=2)


## 10. Load Experiment Logs

In [ ]:
def load_histories(run_dirs):
    frames = []
    for run_dir in run_dirs:
        csv_path = Path(run_dir) / 'log_history.csv'
        if csv_path.exists():
            frames.append(pd.read_csv(csv_path))
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

history_df = load_histories([RUNS_DIR / 'exp-001', RUNS_DIR / 'exp-002'])
history_df.head() if not history_df.empty else history_df

## 11. Plot Training Curves

In [ ]:
if not history_df.empty:
    plot_df = history_df.dropna(subset=['loss', 'step'], how='all').copy()
    if 'loss' in plot_df.columns:
        plt.figure(figsize=(10, 5))
        sns.lineplot(data=plot_df.dropna(subset=['loss']), x='step', y='loss', hue='experiment')
        plt.title('Training Loss Comparison')
        plt.tight_layout()
        plt.savefig(RUNS_DIR / 'training_loss_comparison.png', dpi=150)
        plt.show()

    if 'eval_loss' in history_df.columns:
        eval_plot_df = history_df.dropna(subset=['eval_loss']).copy()
        if not eval_plot_df.empty:
            plt.figure(figsize=(10, 5))
            sns.lineplot(data=eval_plot_df, x='epoch', y='eval_loss', hue='experiment', marker='o')
            plt.title('Eval Loss Comparison')
            plt.tight_layout()
            plt.savefig(RUNS_DIR / 'eval_loss_comparison.png', dpi=150)
            plt.show()


## 12. Qualitative Sample Evaluation

In [ ]:
sample_prompts = [
    '请用一句话解释什么是监督微调。',
    '把下面这句话改写得更正式：这个结果还不错。',
    '请把以下内容整理成三点：先准备数据，再选择模型，最后开始训练。',
]
sample_prompts

In [ ]:
# After training, load adapters and compare outputs manually.
# This section is intentionally left as a structured placeholder for the first iteration.

## 13. Summary

这个 notebook 的第一版目标不是一次把所有细节做满，而是把项目训练入口、对比实验和结果图像的骨架搭起来。

下一步可以继续补强：
1. 增加 4-bit QLoRA 版本
2. 自动生成样本对比报告
3. 输出实验配置表和结果摘要表
